# North pole star, 3200 BCE – 2026 CE

Under precession of the equinoxes, the north celestial pole traces a circle on sky over roughly 25,800 years, so the star that marks north changes with epoch. The code below finds the naked-eye star nearest to pole at each epoch from 3200 BCE to 2026 CE.

A star's ecliptic latitude $\beta_S$ is effectively fixed over this span, while the celestial pole lies at angular distance $\varepsilon$, the obliquity of the ecliptic, from the ecliptic pole. The pole therefore sits at ecliptic latitude $\beta_P = 90^\circ - \varepsilon$ and longitude $\lambda_P = 90^\circ$, and transforming the star to the equator of date gives its pole distance

$$p_S(T) = 90^\circ - \delta_S(T), \qquad \sin\delta_S = \sin\beta_S\cos\varepsilon + \cos\beta_S\sin\varepsilon\sin\lambda_S .$$

As precession advances $\lambda_S$, the pole distance reaches its minimum over the cycle at $\lambda_S = 90^\circ$,

$$p_{\min}(S) = \lvert \beta_S - \beta_P \rvert ,$$

so a star can come within a tolerance $\tau$ of the pole only if $\lvert \beta_S - \beta_P\rvert \le \tau$.

Positions are evaluated of date, including precession, nutation and proper motion, with the Swiss Ephemeris. The start epoch (3200 BCE, JD $\approx$ 552805) falls below the Moshier ephemeris floor near 3000 BCE, so the DE431 long-range files are required, as listed in the setup cell.

In [1]:
# Requirements:
#   pip install pyswisseph
#   EPHE_PATH below must contain BOTH:
#     - sefstars.txt                      (Swiss Ephemeris fixed-star catalogue)
#     - the DE431 long-range planet+moon files spanning the date range:
#         seplm36 seplm30 seplm24 seplm18 seplm12 seplm06 sepl_00 sepl_06 sepl_12 sepl_18
#         semom36 semom30 semom24 semom18 semom12 semom06 semo_00 semo_06 semo_12 semo_18
#     all from  github.com/aloistr/swisseph -> ephe/   (each ~0.5 MB).

import math
import swisseph as swe

EPHE_PATH = "ephe"       # <-- directory with sefstars.txt AND the DE431 se1 files
swe.set_ephe_path(EPHE_PATH)

FLG_EC = swe.FLG_SWIEPH                          # ecliptic lon/lat of date
# (equatorial is derived analytically, so no equatorial Swiss call is needed)

TAU        = 5.0   # max pole distance to count as a pole star (deg)
TAU_STRICT = 1.0   # strict threshold, annotation only
M_LIM      = 4.0   # magnitude gate (see notes below)

In [2]:
def jd_of_year(astro_year):
    """Julian Day at mid-year (1 Jul, 12h); Julian calendar before 1583."""
    cal = swe.GREG_CAL if astro_year >= 1583 else swe.JUL_CAL
    return swe.julday(astro_year, 7, 1, 12.0, cal)

def year_label(astro_year):
    """Astronomical year -> BCE/CE label (astro 0 = 1 BCE, astro -3199 = 3200 BCE)."""
    return f"{1 - astro_year} BCE" if astro_year <= 0 else f"{astro_year} CE"

def obliquity(jd):
    """True obliquity of the ecliptic of date (deg)."""
    return swe.calc_ut(jd, swe.ECL_NUT)[0][0]

def dec_from_ecliptic(lam, beta, eps):
    """Declination from ecliptic (lam, beta) and obliquity eps, all in degrees:
       sin(dec) = sin(beta) cos(eps) + cos(beta) sin(eps) sin(lam)."""
    r = math.radians
    arg = (math.sin(r(beta)) * math.cos(r(eps)) +
           math.cos(r(beta)) * math.sin(r(eps)) * math.sin(r(lam)))
    arg = max(-1.0, min(1.0, arg))    # guard tiny FP overshoot very near the pole
    return math.degrees(math.asin(arg))

GREEK = {"al": "\u03b1", "be": "\u03b2", "ga": "\u03b3", "de": "\u03b4",
         "ep": "\u03b5", "ze": "\u03b6", "et": "\u03b7", "th": "\u03b8",
         "io": "\u03b9", "ka": "\u03ba", "la": "\u03bb", "mu": "\u03bc",
         "nu": "\u03bd", "xi": "\u03be", "om": "\u03bf", "pi": "\u03c0",
         "rh": "\u03c1", "si": "\u03c3", "ta": "\u03c4", "up": "\u03c5",
         "ph": "\u03c6", "ch": "\u03c7", "ps": "\u03c8", "og": "\u03c9"}

def bayer_designation(code):
    """Swiss Ephemeris Bayer/Flamsteed code -> readable designation:
       'alUMi' -> 'alpha UMi', '11UMi' -> '11 UMi', 'HR4609' -> 'HR 4609'."""
    if len(code) >= 4 and code[-3:].isalpha():
        prefix, const = code[:-3], code[-3:]
        if prefix in GREEK:
            return GREEK[prefix] + " " + const
        if prefix.isdigit():
            return prefix + " " + const
    if code.startswith("HR") and code[2:].isdigit():
        return "HR " + code[2:]
    return code

def load_catalog(path, m_lim):
    """[(lookup, display, mag), ...] for stars with magnitude <= m_lim.
       lookup is what fixstar2_ut needs: the traditional name, or ',Bayer'
       (leading comma) when there is no traditional name, since Swiss Ephemeris
       only searches the Bayer field for comma-prefixed queries -- without the
       comma ~40% of the catalogue silently fails to match.
       display appends the official Bayer designation, e.g. 'Thuban (alpha Dra)'."""
    seen, stars = set(), []
    with open(path, encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 14:
                continue
            trad, bayer = parts[0], parts[1]
            try:
                mag = float(parts[13])
            except ValueError:
                continue
            desig   = bayer_designation(bayer)
            lookup  = trad if trad else ("," + bayer)
            display = f"{trad} ({desig})" if trad else desig
            if mag <= m_lim and lookup not in seen:
                seen.add(lookup)
                stars.append((lookup, display, mag))
    return stars

In [3]:
def star_ecliptic(name, jd):
    """ONE Swiss Ephemeris call: returns (lambda, beta) of date in degrees.
       Declination, and hence pole distance, is then computed analytically,
       so there is no second (equatorial) ephemeris call per star."""
    x = swe.fixstar2_ut(name, jd, FLG_EC)[0]
    return x[0], x[1]

## Selection

Among stars bright enough to see ($m_S \le m_{\text{lim}}$), the code selects one nearest to  pole at time $T$, with brightness breaking ties:

$$S = \operatorname*{arg\,min}_{m_S \le m_{\text{lim}}} p_S(T) \qquad (\text{ties broken by smaller } m_S).$$

$S$ is pole star when $p_S(T) \le \tau$. Otherwise the epoch is a gap, with no true pole star, and $S$ is reported as nearest star for reference.

Since $p_S(T) \ge \lvert \beta_S - \beta_P\rvert$, a star can fall within $\tau$ of pole only if $\lvert \beta_S - \beta_P\rvert \le \tau$, a thin band of ecliptic latitude. So, the pole star, when it exists, is the star of that band nearest the pole, brightest if tied. The nearest star during a gap can lie outside the band, so the search is run over a wider window $50^\circ \le \beta_S \le 83^\circ$ about $\beta_P \approx 66.6^\circ$. This window contains the nearest star at every epoch in range and reproduces an exhaustive catalogue search exactly, at a fraction of the cost.

In [4]:
_J2000 = jd_of_year(2000)
PRELIM = []
for _lookup, _display, _mag in load_catalog(EPHE_PATH.rstrip("/") + "/sefstars.txt", 6.0):
    try:
        _lam, _beta = star_ecliptic(_lookup, _J2000)
    except Exception:
        continue
    if 50.0 <= _beta <= 83.0:
        PRELIM.append((_lookup, _display, _mag))
len(PRELIM)   # working-set size

146

In [5]:
def pole_star_at(astro_year, tau=TAU, m_lim=M_LIM):
    """Return (best, beta_P) where best = (p_S(T), mag, display, beta_S) is the
       visible star (m <= m_lim) closest to the pole at time T, brightness
       breaking exact ties. The caller declares a pole star iff best p_S(T) <= tau."""
    jd     = jd_of_year(astro_year)
    eps    = obliquity(jd)
    beta_P = 90.0 - eps
    best = None
    for lookup, display, mag in PRELIM:
        if mag > m_lim:
            continue
        try:
            lam, beta = star_ecliptic(lookup, jd)          # one call
        except Exception:
            continue
        p = 90.0 - dec_from_ecliptic(lam, beta, eps)       # analytic, no 2nd call
        if best is None or (p, mag) < (best[0], best[1]):  # nearest; mag breaks ties
            best = (p, mag, display, beta)
    return best, beta_P

## Parameters 

$\tau = 5^\circ$ is taken as largest separation at which a star still serves as a pole marker, Vega & Deneb are described as pole stars at about $5^\circ$. The strict sense, within which Polaris appears fixed to unaided eye, is $\tau \approx 1^\circ$, and epochs meeting it are flagged separately.

The magnitude gate $m_{\text{lim}} = 4.0$ keeps Thuban ($3.68$) & Polaris ($2.02$) but drops fainter stars that, although still within naked-eye visibility (Weaver, 1947 places detection near magnitude $6$), are too dim to act as markers. It is a practical cutoff set just above Thuban, the faintest star generally recognised as a former pole star.

Each row gives nearest star, with its proper name followed by the official Bayer designation in brackets, its pole distance $p_S(T)$, its magnitude, and whether the epoch has a pole star, a strict ($\le 1^\circ$) pole star, or no pole star.

In [6]:
years = sorted(set([-3199] + list(range(-2999, 2027, 250)) + [2026]))

print(f"North pole star   (tau = {TAU:.0f} deg,  m_lim = {M_LIM:.1f})\n")
print(f"{'year':>9} | {'star':19} | {'p_S(T)':>8} | {'mag':>5} | status")
print("-" * 74)
for ay in years:
    best, beta_P = pole_star_at(ay)
    if best is None:
        print(f"{year_label(ay):>9} | {'-- none --':19} | {'':>8} | {'':>5} | no visible star")
        continue
    p, mag, display, beta = best
    if   p <= TAU_STRICT: status = "strict pole star (<= 1 deg)"
    elif p <= TAU:        status = "pole star"
    else:                 status = "gap: no pole star (nearest shown)"
    print(f"{year_label(ay):>9} | {display:19} | {p:6.2f} d | {mag:5.2f} | {status}")

North pole star   (tau = 5 deg,  m_lim = 4.0)

     year | star                |   p_S(T) |   mag | status
--------------------------------------------------------------------------
 3200 BCE | Thuban (α Dra)      |   2.28 d |  3.68 | pole star
 3000 BCE | Thuban (α Dra)      |   1.15 d |  3.68 | pole star
 2750 BCE | Thuban (α Dra)      |   0.28 d |  3.68 | strict pole star (<= 1 deg)
 2500 BCE | Thuban (α Dra)      |   1.68 d |  3.68 | pole star
 2250 BCE | Thuban (α Dra)      |   3.09 d |  3.68 | pole star
 2000 BCE | Thuban (α Dra)      |   4.49 d |  3.68 | pole star
 1750 BCE | Ketu (κ Dra)        |   5.41 d |  3.89 | gap: no pole star (nearest shown)
 1500 BCE | Ketu (κ Dra)        |   4.83 d |  3.89 | pole star
 1250 BCE | Ketu (κ Dra)        |   4.68 d |  3.89 | pole star
 1000 BCE | Ketu (κ Dra)        |   5.03 d |  3.89 | gap: no pole star (nearest shown)
  750 BCE | Ketu (κ Dra)        |   5.76 d |  3.89 | gap: no pole star (nearest shown)
  500 BCE | Ketu (κ Dra)        |  